# Zajęcia 7: odczyt wersji i lineage

Notebook nie trenuje i nie zmienia aliasów. Wskaż aktualny plik `evidence.json`
przez `BANK_MARKETING_REGISTRY_EVIDENCE`. Nie jest wybierany najnowszy katalog
ani historyczny run. Dowody wykonań z championem i bez niego są w raporcie zajęć 7.

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from src.registry import LocalRegistry, read_json  # noqa: E402

selected = os.environ.get("BANK_MARKETING_REGISTRY_EVIDENCE")
if not selected:
    raise ValueError("Ustaw BANK_MARKETING_REGISTRY_EVIDENCE na aktualny evidence.json")
evidence_path = Path(selected)
if not evidence_path.is_absolute():
    evidence_path = root / evidence_path
evidence = read_json(evidence_path)
store = LocalRegistry.from_evidence(evidence_path)
version = str(evidence["model_version"])
state_before = store.snapshot()
print({"version": version, "aliases": state_before["aliases"]})

## Rejestracja i jawny wybór batcha

`python -m scripts.verify_lesson7 --rebuild --execute-notebook` odtwarza eksperymenty
i przekazuje konkretny wygenerowany batch do nowego rejestru. Komenda wypisuje
`EVIDENCE=...`. Alternatywnie `--batch <katalog>` wybiera konkretny batch.
Rejestracja nie promuje championa. Notebook ładuje wersję zapisaną w przekazanym dowodzie.

In [ ]:
origin = store.lineage(version)
print({key: origin[key] for key in ("model_name", "model_version", "packaging_run_id")})
print({key: origin["lineage"][key] for key in ("source_run_id", "git_sha", "raw_sha256")})
print({alias: store.resolve(alias) for alias in state_before["aliases"]})

## Jakość na validation

AP musi przewyższać Dummy; recall musi spełniać regułę protokołu.
Jeśli champion istnieje, porównanie wymaga zgodnych danych i konfiguracji.
Kandydat gorszy od championa może być odczytany, lecz jego promocja jest blokowana.

In [ ]:
try:
    decision = store.check(version)
except (ValueError, KeyError, TypeError) as error:
    decision = {"eligible": False, "reason": str(error)}
print(decision)
print(
    {"dummy_ap": origin["lineage"]["dummy_ap"], "recall_floor": origin["lineage"]["recall_floor"]}
)

## Model i nienaruszony stan rejestru

Ładujemy kompletny Pipeline. Sprawdzamy stan zastany; champion może istnieć
lub nie istnieć. Final test i replay nie są oceniane.

In [ ]:
model = store.load(version)
print(type(model).__name__)
print(origin["lineage"]["schema"])
assert store.snapshot() == state_before
print("PASS: aliasy i wersje rejestru pozostały niezmienione")

## Promocja i rollback

Zmiana championa wymaga osobnej zgody operatora i zgodności oczekiwanej wersji.
Ten notebook nie wywołuje `promote` ani `rollback`. Osobna demonstracja UCI
używa izolowanej bazy; testy sprawdzają także blokady jakości i uszkodzenia pakietu.
Recall 7/794 jest ograniczeniem modelu akademickiego, a nie dowodem gotowości produkcyjnej.